# Type-constrained generation — LC-QuAD eval on Colab GPU

Runs `test.py` (all 1000 LC-QuAD test questions through the constrained generator) on a Colab GPU.

**One-time setup before running:**

1. Upload two files to a Google Drive folder (default expected: `MyDrive/ogd/`):
   - `qwen_lcquad.safetensors` (3.1 GB) — from your local `model/` folder
   - `class_tries.pkl` (197 MB) — from your local `dbpedia/` folder
   - (`entities.pkl` is NOT needed — generation only reads the tries)
2. Add a GitHub token as a Colab secret (repo is private): left sidebar → key icon → Secrets → name `GITHUB_TOKEN`.
   - Alternatively make the repo public and comment out the token line in the clone cell.
3. Runtime → Change runtime type → GPU.

Then Runtime → Run all. output.json is mirrored to Drive every 2 minutes while the eval runs, and the final numbers print at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # re-running this cell fixes dropped Drive connections

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ogd')  # where you uploaded the two files
REPO_DIR = Path('/content/ontologically-guided-decoding')

assert (DRIVE_DIR / 'qwen_lcquad.safetensors').exists(), 'weights not found in Drive folder'
assert (DRIVE_DIR / 'class_tries.pkl').exists(), 'class_tries.pkl not found in Drive folder'

Mounted at /content/drive


In [ ]:
!git clone https://github.com/JosephMuddle/ontologically-guided-decoding.git {REPO_DIR}
!git -C /content/ontologically-guided-decoding pull


fatal: destination path '/content/ontologically-guided-decoding' already exists and is not an empty directory.
Already up to date.


In [ ]:
import shutil
(REPO_DIR / 'model').mkdir(exist_ok=True)
(REPO_DIR / 'dbpedia').mkdir(exist_ok=True)
shutil.copy(DRIVE_DIR / 'qwen_lcquad.safetensors', REPO_DIR / 'model' / 'qwen_lcquad.safetensors')
shutil.copy(DRIVE_DIR / 'class_tries.pkl', REPO_DIR / 'dbpedia' / 'class_tries.pkl')
print('weights and tries copied to VM local disk')

# Resume support. test.py reads output.json from the REPO directory on this VM,
# never from Drive -- Drive is only where cell 6 mirrors it so a dropped session
# does not lose the run. So deleting the Drive copy on its own resets nothing,
# and the mirror will put it straight back. This cell is the one place the two
# are reconciled, so re-run it (not just the eval cell) whenever you change your
# mind about resuming.
RESUME = True
out = REPO_DIR / 'output.json'
drive_out = DRIVE_DIR / 'output.json'
if not RESUME:
    for f in (out, drive_out):
        if f.exists():
            f.unlink()
    print('RESUME is False: cleared both copies, starting from question 1')
elif drive_out.exists():
    import json
    shutil.copy(drive_out, out)
    n = len(json.loads(out.read_text(encoding='utf-8')))
    print(f'resuming: {n} records restored from Drive, {1000 - n} questions left')
elif out.exists():
    import json
    n = len(json.loads(out.read_text(encoding='utf-8')))
    print(f'no copy in Drive, but this VM already holds {n} records -- resuming from those.')
    print('set RESUME = False and re-run this cell to start from question 1 instead')
else:
    print('no output.json on the VM or in Drive; starting from question 1')

In [ ]:
# .env holds only paths here (no secrets); config and tokenizer come from the
# public base model Qwen/Qwen2.5-Coder-1.5B, only the weights are local
(REPO_DIR / '.env').write_text('MODEL_WEIGHTS=model/qwen_lcquad.safetensors\nDATA_PATH=dbpedia\n')
!pip install -q xgrammar safetensors
print('deps installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 52.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
deps installed


In [ ]:
import torch, time
print('GPU:', torch.cuda.get_device_name(0))
%cd {REPO_DIR}
t0 = time.time()
from type_constrained_generation import generate  # module-level: loads weights, grammars, tries
print(f'setup took {time.time() - t0:.0f}s')
print(generate('What is the region of Tom Perriello ?'))  # smoke test: one constrained query

GPU: NVIDIA A100-SXM4-80GB
/content/ontologically-guided-decoding


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

loaded qwen_lcquad.safetensors into Qwen/Qwen2.5-Coder-1.5B on cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

compiled grammars: 5 beginning templates, 616 relations + type tail, 763 classes
loaded 762 class tries in 10.2s
setup took 116s
SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Tom_Perriello> <http://dbpedia.org/ontology/region> ?uri }


In [ ]:
# mirror output.json to Drive every 2 min so progress survives a disconnect
!nohup bash -c 'while true; do cp -f /content/ontologically-guided-decoding/output.json /content/drive/MyDrive/ogd/output.json 2>/dev/null; sleep 120; done' > /dev/null 2>&1 &
print('mirroring output.json to Drive every 2 min')

mirroring output.json to Drive every 2 min


In [ ]:
# The eval. By default all four rungs of the ladder; ONLY narrows it. After a
# decoder change that affects one rung -- say the boosts -- set ONLY to that rung
# and REDO to True: test.py clears just those fields from output.json, regenerates
# them, and leaves the other three exactly as they were. Without REDO a rung that
# already has answers is skipped, which is what makes a timed-out run resumable.
BEAMS = 7      # whole-query beam width: every rung and the baseline (1 = greedy)
ONLY = ''      # '' = all four; else e.g. 'generated' or 'generated no_boosts'
REDO = False   # True to regenerate the selected rungs where answers already exist

args = f'--beams {BEAMS}'
if ONLY:
    args += f' --systems {ONLY}'
if REDO:
    args += ' --redo'
print('test.py', args)
!python /content/ontologically-guided-decoding/test.py {args}

In [ ]:
import json, shutil
shutil.copy(REPO_DIR / 'output.json', DRIVE_DIR / 'output.json')  # final copy
res = json.loads((REPO_DIR / 'output.json').read_text(encoding='utf-8'))

# the ablation ladder, strongest first: generated - no_boosts isolates the
# ontological boosts, no_boosts - grammar_only isolates the KB vocabulary
for name in ('generated', 'no_boosts', 'grammar_only', 'unconstrained'):
    have = [r for r in res if f'{name}_match' in r]   # a rung may not have been run
    if not have:
        print(f'{name:14} not run')
        continue
    hits = sum(r[f'{name}_match'] for r in have)
    print(f'{name:14} {hits}/{len(have)} ({hits / len(have):.1%})')
twin = sum(r['match_modulo_twins'] for r in res)
print(f'{"mod-twins":14} {twin}/{len(res)} ({twin / len(res):.1%})  (full system only)')

In [ ]:
!git -C /content/ontologically-guided-decoding log --oneline -1
!grep -c "bitmask.to(DEVICE)" /content/ontologically-guided-decoding/type_constrained_generation.py

86e1540 (HEAD -> main, origin/main, origin/HEAD) UNCONSTRAINED BASELINE
2


In [ ]:
!git -C /content/ontologically-guided-decoding pull


Already up to date.
